In [ ]:
import pandas as pd
import numpy as np
import re


In [ ]:
# ============================================
# CARGA DE DATOS
# ============================================
GOOGLE_DRIVE_FILE_ID = "ID"  # Reemplazar con el ID real
url = f"https://drive.usercontent.google.com/download?id={GOOGLE_DRIVE_FILE_ID}"

try:
    df1 = pd.read_excel(url, index_col=0)
    print(f"✅ Archivo cargado: {len(df1)} registros")
except Exception as e:
    print(f"❌ Error al cargar archivo: {e}")
    print("💡 Intentando cargar desde archivo local...")
    # Intenta cargar desde archivo local si falla
    # df1 = pd.read_excel("ruta/al/archivo.xlsx", index_col=0)

# Eliminar columnas innecesarias (si existen)
columnas_a_eliminar = [
    'Attribute Name (pa_acabado)', 'Attribute Value (pa_acabado)',
    'Attribute Name (pa_cantos_bordes)', 'Attribute Value (pa_cantos_bordes)',
    'Attribute Name (pa_caras_destonalizado)', 'Attribute Value (pa_caras_destonalizado)',
    'Attribute Name (pa_estetica_diseno)', 'Attribute Value (pa_estetica_diseno)',
    'Attribute Name (pa_formato)', 'Attribute Value (pa_formato)',
    'Attribute Name (pa_mÂ²_por_caja)', 'Attribute Value (pa_mÂ²_por_caja)',
    'Attribute Name (pa_marcas)', 'Attribute Value (pa_marcas)',
    'Attribute Name (pa_terminacion)', 'Attribute Value (pa_terminacion)'
]

columnas_existentes = [col for col in columnas_a_eliminar if col in df1.columns]
if columnas_existentes:
    df2 = df1.drop(columnas_existentes, axis=1)
else:
    df2 = df1.copy()

print(f"📋 Columnas del DataFrame: {list(df2.columns)}")

# ============================================
# FUNCIONES DE LIMPIEZA
# ============================================

def limpiar_texto_para_excel(texto, max_chars=32000):
    """
    Limpia texto para que sea compatible con Excel:
    - Elimina caracteres ilegales
    - Trunca si excede el límite
    - Limpia HTML problemático
    """
    if pd.isna(texto):
        return ""

    texto_str = str(texto)

    # Reemplazar caracteres problemáticos
    caracteres_problematicos = {
        '\x00': '',  # NULL
        '\x01': '',
        '\x02': '',
        '\x03': '',
        '\x04': '',
        '\x05': '',
        '\x06': '',
        '\x07': '',
        '\x08': '',
        '\x0b': '',
        '\x0c': '',
        '\x0e': '',
        '\x0f': '',
        '\x10': '',
        '\x11': '',
        '\x12': '',
        '\x13': '',
        '\x14': '',
        '\x15': '',
        '\x16': '',
        '\x17': '',
        '\x18': '',
        '\x19': '',
        '\x1a': '',
        '\x1b': '',
        '\x1c': '',
        '\x1d': '',
        '\x1e': '',
        '\x1f': '',
    }

    for char, replacement in caracteres_problematicos.items():
        texto_str = texto_str.replace(char, replacement)

    # Truncar si es muy largo
    if len(texto_str) > max_chars:
        texto_str = texto_str[:max_chars] + "... [TRUNCADO]"

    return texto_str

# ============================================
# FUNCIONES DE EXTRACCIÓN Y COMPARACIÓN
# ============================================

def extraer_medida_producto(titulo):
    """
    Extrae la medida en formato NúmeroXNúmero del título del producto.
    Limpia cualquier símbolo extraño y retorna solo los números.

    Ejemplos:
        'Cerámica Blanca 60x60(1,44)' -> '60x60'
        'Malla 30X30? Ocre' -> '30x30'
        'Piso 120x60! Premium' -> '120x60'
        'Producto Sin Medida' -> None
    """
    if pd.isna(titulo):
        return None

    # Patrón: busca números seguidos de x/X y más números
    patron = r'(\d+)\s*[xX×]\s*(\d+)'

    match = re.search(patron, str(titulo))

    if match:
        num1 = match.group(1)
        num2 = match.group(2)
        medida = f"{num1}x{num2}"
        return medida

    return None


def buscar_medida_en_content(content, medida):
    """
    Busca la medida exacta en el contenido del producto.
    Retorna True si encuentra coincidencia exacta.
    """
    if pd.isna(content) or pd.isna(medida):
        return False

    content_str = str(content).lower()
    medida_lower = medida.lower()

    # Crear variaciones de búsqueda
    num1, num2 = medida_lower.split('x')

    patrones = [
        medida_lower,                      # 60x60
        f"{num1}X{num2}",                  # 60X60
        f"{num1} x {num2}",                # 60 x 60
        f"{num1}  x  {num2}",              # 60  x  60
        f"{num1}×{num2}",                  # 60×60
        f"{num1} × {num2}",                # 60 × 60
    ]

    # Buscar cualquiera de los patrones
    for patron in patrones:
        if patron.lower() in content_str:
            return True

    return False


def extraer_medida_de_content(content):
    """
    Extrae la primera medida encontrada en el Content.
    """
    if pd.isna(content):
        return None

    patron = r'(\d+)\s*[xX×]\s*(\d+)'
    match = re.search(patron, str(content))

    if match:
        num1 = match.group(1)
        num2 = match.group(2)
        return f"{num1}x{num2}"

    return None


# ============================================
# PROCESAMIENTO
# ============================================

print("\n" + "="*60)
print("🔍 EXTRAYENDO MEDIDAS DE LOS PRODUCTOS")
print("="*60)

# Limpiar Content para evitar errores de Excel
print("\n🧹 Limpiando contenido para compatibilidad con Excel...")
df2['Content_Original'] = df2['Content'].copy()
df2['Content'] = df2['Content'].apply(limpiar_texto_para_excel)

# Extraer medida del título
df2['Medida_Producto'] = df2.index.to_series().apply(extraer_medida_producto)

# Extraer medida del content
df2['Medida_Content'] = df2['Content'].apply(extraer_medida_de_content)

# Buscar si la medida del producto está en el content
df2['Medida_Coincide'] = df2.apply(
    lambda row: buscar_medida_en_content(row['Content'], row['Medida_Producto']),
    axis=1
)

# Verificar si las medidas son exactamente iguales
df2['Medida_Exacta'] = df2.apply(
    lambda row: (row['Medida_Producto'] == row['Medida_Content'])
    if (pd.notna(row['Medida_Producto']) and pd.notna(row['Medida_Content']))
    else False,
    axis=1
)

# ============================================
# ESTADÍSTICAS Y RESULTADOS
# ============================================

total = len(df2)
con_medida_producto = df2['Medida_Producto'].notna().sum()
con_medida_content = df2['Medida_Content'].notna().sum()
coincidencias = df2['Medida_Coincide'].sum()
exactas = df2['Medida_Exacta'].sum()
sin_medida = total - con_medida_producto

print(f"\n📊 ESTADÍSTICAS:")
print(f"   Total de productos: {total}")
print(f"   Con medida en Título: {con_medida_producto} ({con_medida_producto/total*100:.1f}%)")
print(f"   Con medida en Content: {con_medida_content} ({con_medida_content/total*100:.1f}%)")
print(f"   Medidas que coinciden: {coincidencias} ({coincidencias/total*100:.1f}%)")
print(f"   Medidas exactamente iguales: {exactas} ({exactas/total*100:.1f}%)")
print(f"   Sin medida en título: {sin_medida} ({sin_medida/total*100:.1f}%)")

# Mostrar casos de interés
print("\n" + "="*60)
print("✅ PRODUCTOS CON COINCIDENCIA DE MEDIDAS")
print("="*60)
coinciden = df2[df2['Medida_Coincide'] == True]
if len(coinciden) > 0:
    print(f"Total: {len(coinciden)} productos")
    for idx, row in coinciden.head(5).iterrows():
        print(f"\n📦 {idx}")
        print(f"   Medida: {row['Medida_Producto']}")
        print(f"   SKU: {row['SKU']}")
else:
    print("   No se encontraron coincidencias")

print("\n" + "="*60)
print("⚠️  PRODUCTOS SIN COINCIDENCIA (Medida en título pero no en Content)")
print("="*60)
sin_coincidencia = df2[
    (df2['Medida_Producto'].notna()) &
    (df2['Medida_Coincide'] == False)
]
if len(sin_coincidencia) > 0:
    print(f"Total: {len(sin_coincidencia)} productos")
    for idx, row in sin_coincidencia.head(5).iterrows():
        print(f"\n📦 {idx}")
        print(f"   Medida esperada: {row['Medida_Producto']}")
        print(f"   Medida en Content: {row['Medida_Content'] or 'No encontrada'}")
        print(f"   SKU: {row['SKU']}")
else:
    print("   ¡Todas las medidas coinciden! ✨")

print("\n" + "="*60)
print("❌ PRODUCTOS SIN MEDIDA EN TÍTULO")
print("="*60)
sin_medida_df = df2[df2['Medida_Producto'].isna()]
if len(sin_medida_df) > 0:
    print(f"Total: {len(sin_medida_df)} productos")
    for idx, row in sin_medida_df.head(5).iterrows():
        print(f"\n📦 {idx}")
        print(f"   SKU: {row['SKU']}")
        print(f"   Medida en Content: {row['Medida_Content'] or 'No encontrada'}")
else:
    print("   Todos los productos tienen medida")

# ============================================
# EXPORTACIÓN SEGURA
# ============================================

print("\n" + "="*60)
print("💾 EXPORTANDO RESULTADOS")
print("="*60)

# Preparar DataFrame para exportar
columnas_exportar = ['SKU', 'Medida_Producto', 'Medida_Content',
                     'Medida_Coincide', 'Medida_Exacta']

# Filtrar solo las columnas que existen
columnas_exportar = [col for col in columnas_exportar if col in df2.columns]

df_export = df2[columnas_exportar].copy()

try:
    # Exportar con motor openpyxl que maneja mejor los caracteres especiales
    df_export.to_excel('MEDIDAS_ANALISIS.xlsx', index=True, engine='openpyxl')
    print(f"✅ Archivo Excel exportado: MEDIDAS_ANALISIS.xlsx")
except Exception as e:
    print(f"⚠️  Error al exportar Excel: {e}")
    print("   Intentando con formato CSV...")

# Siempre exportar CSV como backup
try:
    df_export.to_csv('MEDIDAS_ANALISIS.csv', index=True, encoding='utf-8-sig')
    print(f"✅ Archivo CSV exportado: MEDIDAS_ANALISIS.csv")
except Exception as e:
    print(f"❌ Error al exportar CSV: {e}")

# Exportar solo productos con problemas
df_problemas = df2[
    (df2['Medida_Producto'].notna()) &
    (df2['Medida_Coincide'] == False)
][columnas_exportar].copy()

if len(df_problemas) > 0:
    try:
        df_problemas.to_excel('MEDIDAS_PROBLEMAS.xlsx', index=True, engine='openpyxl')
        print(f"⚠️  Productos con problemas: MEDIDAS_PROBLEMAS.xlsx ({len(df_problemas)} productos)")
    except Exception as e:
        print(f"⚠️  Error exportando problemas a Excel, usando CSV...")
        df_problemas.to_csv('MEDIDAS_PROBLEMAS.csv', index=True, encoding='utf-8-sig')
        print(f"⚠️  Productos con problemas: MEDIDAS_PROBLEMAS.csv ({len(df_problemas)} productos)")

# Resumen final
print("\n" + "="*60)
print("📋 RESUMEN DE ARCHIVOS GENERADOS")
print("="*60)
print("1. MEDIDAS_ANALISIS.xlsx/.csv - Análisis completo")
print("2. MEDIDAS_PROBLEMAS.xlsx/.csv - Solo productos con discrepancias")

print("\n✨ Análisis completado exitosamente!")
print("\n💡 INTERPRETACIÓN:")
print(f"   - {coincidencias} productos tienen la medida correctamente documentada")
print(f"   - {len(sin_coincidencia)} productos necesitan actualizar su descripción")
print(f"   - {sin_medida} productos no tienen medida en el título")